# 01 - Data Preparation & Exploratory Data Analysis

## Objetivo
En este notebook se realiza:

- Extracción de datos desde la fuente oficial (MIDAGRI)
- Limpieza y transformación de la serie temporal
- Validación de la calidad de datos
- Análisis exploratorio de la serie
- Identificación de tendencia, estacionalidad y patrones
- Análisis de autocorrelación (ACF y PACF)

Este paso es fundamental para entender la estructura de la serie antes del modelado.

In [1]:
## Fuente
# Portal SIEA - MIDAGRI:
# https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo

In [5]:
# Librerías base
import re
import io
import os
import time
import requests
import warnings
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs

# Manejo de datos
import numpy as np
import pandas as pd

# Parsing HTML
from bs4 import BeautifulSoup

# Lectura de PDF
import pdfplumber

# Visualización
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [6]:
SITE_URL = "https://siea.midagri.gob.pe"
BASE_URL = "https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo"

YEAR_URLS = [
    "https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/468-2025",
    "https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/498-2026"
]

### Extraer meses

In [12]:
# Aquí guardaremos todos los meses
all_month_links = []

# Mozilla/5. es un User-Agent que ayuda a que la web no bloquee la solicitud, para q no nos salga error 403
headers = {"User-Agent": "Mozilla/5.0"}

# Recorremos cada año (2025 y 2026)
for year_url in YEAR_URLS:
        
    #  Pedimos el HTML de la página
    response = requests.get(year_url, headers=headers)
    
    #  Convertimos a formato BeautifulSoup
    soup = BeautifulSoup(response.text, "html.parser") #beautifulSoup hace q el formato html se pueda leer en python
    
    #  Sacamos todos los links <a href="...">
    links = []
    
    for a in soup.find_all("a"):
        if a.get("href") is not None:
            
            href = a.get("href")
            
            # Convertimos a link completo
            full_link = urljoin(SITE_URL, href)
            
            links.append(full_link)
    
    # Quitamos duplicados
    links = list(set(links))
    
    #  Filtramos solo los meses
    month_links = []
    
    for link in links:
        
        if BASE_URL in link:          # pertenece a la sección de huevo
            if link != year_url:      # no es la página del año
                if "download=" not in link:  # no es descarga
                    month_links.append(link)
    
    
    # Guardamos
    all_month_links.extend(month_links)

# Quitamos duplicados finales
all_month_links = list(set(all_month_links))

In [13]:
df_months = pd.DataFrame({"month_url": all_month_links})

df_months #hay 8 links

,month_url
0,https://siea.midagri.gob.pe/portal/publicacion...
1,https://siea.midagri.gob.pe/portal/publicacion...
2,https://siea.midagri.gob.pe/portal/publicacion...
3,https://siea.midagri.gob.pe/portal/publicacion...
4,https://siea.midagri.gob.pe/portal/publicacion...
5,https://siea.midagri.gob.pe/portal/publicacion...
6,https://siea.midagri.gob.pe/portal/publicacion...
7,https://siea.midagri.gob.pe/portal/publicacion...
8,https://siea.midagri.gob.pe/portal/publicacion...


In [14]:
# Aquí guardaremos todos los registros de todos los meses
all_daily_records = []

# Lista de nombres de meses para detectar fechas en texto
meses = [
    "enero", "febrero", "marzo", "abril", "mayo", "junio",
    "julio", "agosto", "setiembre", "septiembre",
    "octubre", "noviembre", "diciembre"
]

In [15]:
# recorrer mes a mes

for month_url in all_month_links:
    
    print(month_url)
    
    # 1. Descargar el HTML del mes
    response = requests.get(month_url, headers=headers)
    
    # 2. Convertir a BeautifulSoup
    soup = BeautifulSoup(response.text, "html.parser")
    
    # 3. Revisar todos los bloques div(organizan) de la página
    for div in soup.find_all("div"):
        
        texto = div.get_text(" ", strip=True) #extrae el texto dentro de cada div
        
        # 4. Nos quedamos solo con bloques que parecen boletines
        # porque contienen un mes en el texto y además la palabra DESCARGAR
        if any(mes in texto.lower() for mes in meses) and "descargar" in texto.lower():
            
            # 5. Buscar links dentro de ese bloque
            links_locales = []
            
            for a in div.find_all("a"):
                href = a.get("href")
                
                if href is not None:
                    link_completo = urljoin(SITE_URL, href)
                    links_locales.append(link_completo)
            
            # 6. Buscar el link de descarga del PDF
            download_url = None
            
            for link in links_locales:
                if "download=" in link:
                    download_url = link
                    break
            
            # 7. Guardar el registro
            all_daily_records.append({
                "month_url": month_url,
                "texto_boletin": texto,
                "download_url": download_url
            })

https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/468-2025/496-huevo-diciembre
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/468-2025/470-huevo-octubre
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/498-2026/536-huevo-marzo
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/498-2026/524-huevo-febrero
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/468-2025/487-huevo-noviembre
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/498-2026/549-huevo-abril
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/468-2025/469-huevo-set

In [17]:
#convertimos a dataframe
df_daily_catalog = pd.DataFrame(all_daily_records)

In [19]:
df_daily_catalog

,month_url,texto_boletin,download_url
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...
1,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...
2,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...
3,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...
4,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...
...,...,...,...
139,https://siea.midagri.gob.pe/portal/publicacion...,22 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
140,https://siea.midagri.gob.pe/portal/publicacion...,21 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
141,https://siea.midagri.gob.pe/portal/publicacion...,20 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
142,https://siea.midagri.gob.pe/portal/publicacion...,19 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...


In [20]:
print("Total de registros encontrados:", len(df_daily_catalog))
print("Registros con link de descarga:", df_daily_catalog["download_url"].notna().sum())

Total de registros encontrados: 144
Registros con link de descarga: 144


In [21]:
#limpiamos duplicados
df_daily_catalog = df_daily_catalog.drop_duplicates().reset_index(drop=True)

print("Total después de quitar duplicados:", len(df_daily_catalog))

Total después de quitar duplicados: 104


In [22]:
df_daily_catalog

,month_url,texto_boletin,download_url
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...
1,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...
2,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...
3,https://siea.midagri.gob.pe/portal/publicacion...,31 diciembre 2025 Hot DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
4,https://siea.midagri.gob.pe/portal/publicacion...,30 diciembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
...,...,...,...
99,https://siea.midagri.gob.pe/portal/publicacion...,22 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
100,https://siea.midagri.gob.pe/portal/publicacion...,21 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
101,https://siea.midagri.gob.pe/portal/publicacion...,20 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
102,https://siea.midagri.gob.pe/portal/publicacion...,19 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...


Ya tenemos todos los pdfs, ahora normalizaremos los nombres

In [23]:
import re


In [24]:
fechas_extraidas = []

for texto in df_daily_catalog["texto_boletin"]:
    
    texto = str(texto).lower()
    
    patron = r"(\d{1,2}\s+(?:enero|febrero|marzo|abril|mayo|junio|julio|agosto|setiembre|septiembre|octubre|noviembre|diciembre)\s+\d{4})"
    
    match = re.search(patron, texto)
    
    if match:
        fechas_extraidas.append(match.group(1))
    else:
        fechas_extraidas.append(None)

df_daily_catalog["fecha_texto"] = fechas_extraidas

In [25]:
df_daily_catalog

,month_url,texto_boletin,download_url,fecha_texto
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
1,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
2,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
3,https://siea.midagri.gob.pe/portal/publicacion...,31 diciembre 2025 Hot DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,31 diciembre 2025
4,https://siea.midagri.gob.pe/portal/publicacion...,30 diciembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,30 diciembre 2025
...,...,...,...,...
99,https://siea.midagri.gob.pe/portal/publicacion...,22 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,22 enero 2026
100,https://siea.midagri.gob.pe/portal/publicacion...,21 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,21 enero 2026
101,https://siea.midagri.gob.pe/portal/publicacion...,20 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,20 enero 2026
102,https://siea.midagri.gob.pe/portal/publicacion...,19 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,19 enero 2026


In [26]:
#eliminamos filas sin fecha o sin pdf
df_daily_catalog = df_daily_catalog[
    df_daily_catalog["fecha_texto"].notna() &
    df_daily_catalog["download_url"].notna()
].reset_index(drop=True)

print("Total final de boletines válidos:", len(df_daily_catalog))

Total final de boletines válidos: 104


In [27]:
df_daily_catalog

,month_url,texto_boletin,download_url,fecha_texto
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
1,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
2,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
3,https://siea.midagri.gob.pe/portal/publicacion...,31 diciembre 2025 Hot DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,31 diciembre 2025
4,https://siea.midagri.gob.pe/portal/publicacion...,30 diciembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,30 diciembre 2025
...,...,...,...,...
99,https://siea.midagri.gob.pe/portal/publicacion...,22 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,22 enero 2026
100,https://siea.midagri.gob.pe/portal/publicacion...,21 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,21 enero 2026
101,https://siea.midagri.gob.pe/portal/publicacion...,20 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,20 enero 2026
102,https://siea.midagri.gob.pe/portal/publicacion...,19 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,19 enero 2026


In [28]:
!pip install pdfplumber


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


DESARGAR LOS PDFS

In [37]:
import os
os.getcwd()
os.listdir()

['01_data_preparation_and_eda.ipynb']

In [42]:
df_daily_catalog

,month_url,texto_boletin,download_url,fecha_texto
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
1,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
2,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
3,https://siea.midagri.gob.pe/portal/publicacion...,31 diciembre 2025 Hot DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,31 diciembre 2025
4,https://siea.midagri.gob.pe/portal/publicacion...,30 diciembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,30 diciembre 2025
...,...,...,...,...
99,https://siea.midagri.gob.pe/portal/publicacion...,22 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,22 enero 2026
100,https://siea.midagri.gob.pe/portal/publicacion...,21 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,21 enero 2026
101,https://siea.midagri.gob.pe/portal/publicacion...,20 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,20 enero 2026
102,https://siea.midagri.gob.pe/portal/publicacion...,19 enero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,19 enero 2026


In [44]:
import os
pdf_folder = "../pdfs"

In [45]:
import requests

# Recorremos todos los links
for i, row in df_daily_catalog.iterrows():
    
    url = row["download_url"]
    fecha = row["fecha_texto"]
    
    print(f"Descargando {i} - {fecha}")
    
    try:
        response = requests.get(url)
        
        # limpiar nombre del archivo (quitar espacios y caracteres raros)
        fecha_clean = fecha.replace(" ", "_").replace("/", "-")
        
        file_path = f"{pdf_folder}/{fecha_clean}.pdf"
        
        with open(file_path, "wb") as f:
            f.write(response.content)
    
    except Exception as e:
        print("Error en:", fecha, e)

Descargando 0 - 25 diciembre 2025
Descargando 1 - 25 diciembre 2025
Descargando 2 - 25 diciembre 2025
Descargando 3 - 31 diciembre 2025
Descargando 4 - 30 diciembre 2025
Descargando 5 - 29 diciembre 2025
Descargando 6 - 22 diciembre 2025
Descargando 7 - 19 diciembre 2025
Descargando 8 - 18 diciembre 2025
Descargando 9 - 17 diciembre 2025
Descargando 10 - 16 diciembre 2025
Descargando 11 - 15 diciembre 2025
Descargando 12 - 12 diciembre 2025
Descargando 13 - 25 octubre 2025
Descargando 14 - 25 octubre 2025
Descargando 15 - 25 octubre 2025
Descargando 16 - 31 octubre 2025
Descargando 17 - 30 octubre 2025
Descargando 18 - 29 octubre 2025
Descargando 19 - 28 octubre 2025
Descargando 20 - 27 octubre 2025
Descargando 21 - 24 octubre 2025
Descargando 22 - 23 octubre 2025
Descargando 23 - 22 octubre 2025
Descargando 24 - 21 octubre 2025
Descargando 25 - 20 octubre 2025
Descargando 26 - 26 marzo 2026
Descargando 27 - 26 marzo 2026
Descargando 28 - 26 marzo 2026
Descargando 29 - 31 marzo 2026
De

### Leer el pdf